In [1]:
import pandas as pd
import json
import os
from collections import defaultdict, Counter

# Load the ranking CSV
ranking_df = pd.read_csv('../Results/player_rankings_z_score.csv')

# Function to extract positions from all JSON files in the lineups directory
def extract_positions_from_lineups(lineups_dir='../data/wwc2023/lineups'):
    """
    Extract position data for all players from all StatsBomb lineup JSON files
    Returns a dictionary mapping player_id to position info
    """
    player_positions = defaultdict(list)
    
    # Get all JSON files in the lineups directory
    json_files = [f for f in os.listdir(lineups_dir) if f.endswith('.json')]
    
    print(f"Found {len(json_files)} JSON files in {lineups_dir}")
    
    for filename in json_files:
        file_path = os.path.join(lineups_dir, filename)
        
        try:
            with open(file_path, 'r') as f:
                match_data = json.load(f)
            
            # Process each team in the match
            for team in match_data:
                lineup = team.get('lineup', [])
                
                for player in lineup:
                    player_id = player.get('player_id')
                    positions = player.get('positions', [])
                    
                    # Extract all positions played by this player in this match
                    for pos in positions:
                        if pos.get('position'):
                            player_positions[player_id].append({
                                'position': pos['position'],
                                'position_id': pos.get('position_id'),
                                'match_file': filename,
                                'from_time': pos.get('from', '00:00'),
                                'to_time': pos.get('to', 'Full Time'),
                                'start_reason': pos.get('start_reason', ''),
                                'end_reason': pos.get('end_reason', '')
                            })
                            
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    
    print(f"Extracted position data for {len(player_positions)} unique players")
    return player_positions

# Function to determine primary position for each player
def get_primary_positions(player_positions):
    """
    Determine the most common position for each player
    Also track if they played multiple positions
    """
    primary_positions = {}
    
    for player_id, positions in player_positions.items():
        if not positions:
            continue
            
        # Count position frequencies
        position_counts = Counter([pos['position'] for pos in positions])
        
        # Get most common position
        most_common_pos = position_counts.most_common(1)[0][0]
        total_appearances = len(positions)
        most_common_count = position_counts[most_common_pos]
        
        # Check if player has positional versatility
        num_different_positions = len(position_counts)
        
        # Count unique matches played
        unique_matches = len(set([pos['match_file'] for pos in positions]))
        
        primary_positions[player_id] = {
            'primary_position': most_common_pos,
            'position_frequency': f"{most_common_count}/{total_appearances}",
            'matches_played': unique_matches,
            'num_positions': num_different_positions,
            'all_positions': list(position_counts.keys()),
            'position_detail': dict(position_counts)
        }
    
    return primary_positions

# Function to simplify position names
def simplify_position(position):
    """
    Simplify detailed position names to broader categories
    """
    if not position:
        return 'Unknown'
    
    position_mapping = {
        'Goalkeeper': 'Goalkeeper',
        'Left Back': 'Defender', 'Right Back': 'Defender',
        'Left Center Back': 'Defender', 'Right Center Back': 'Defender',
        'Left Wing Back': 'Defender', 'Right Wing Back': 'Defender',
        'Center Defensive Midfield': 'Midfielder',
        'Left Defensive Midfield': 'Midfielder', 'Right Defensive Midfield': 'Midfielder',
        'Left Center Midfield': 'Midfielder', 'Right Center Midfield': 'Midfielder',
        'Left Midfield': 'Midfielder', 'Right Midfield': 'Midfielder',
        'Left Attacking Midfield': 'Midfielder', 'Right Attacking Midfield': 'Midfielder',
        'Center Attacking Midfield': 'Midfielder',
        'Left Wing': 'Forward', 'Right Wing': 'Forward',
        'Left Center Forward': 'Forward', 'Right Center Forward': 'Forward',
        'Center Forward': 'Forward'
    }
    
    return position_mapping.get(position, 'Midfielder')

# Function to get more granular position categories
def get_position_category(position):
    """
    More detailed position categorization
    """
    if not position:
        return 'Unknown'
    
    if 'Goalkeeper' in position:
        return 'Goalkeeper'
    elif any(back in position for back in ['Back', 'Wing Back']):
        return 'Fullback/Wing-Back'
    elif 'Center Back' in position:
        return 'Centre-Back'
    elif 'Defensive Midfield' in position:
        return 'Defensive Midfielder'
    elif 'Center Midfield' in position or position in ['Left Midfield', 'Right Midfield']:
        return 'Central Midfielder'
    elif 'Attacking Midfield' in position:
        return 'Attacking Midfielder'
    elif 'Wing' in position:
        return 'Winger'
    elif 'Forward' in position:
        return 'Forward'
    else:
        return 'Other'

# Main execution
if __name__ == "__main__":
    # Extract position data from all lineup JSON files
    lineups_directory = '../data/wwc2023/lineups'
    
    player_positions = extract_positions_from_lineups(lineups_directory)
    
    # Get primary positions for each player
    primary_positions = get_primary_positions(player_positions)
    
    # Create position dataframe
    position_data = []
    for player_id, pos_info in primary_positions.items():
        position_data.append({
            'player_id': float(player_id),
            'primary_position': pos_info['primary_position'],
            'simplified_position': simplify_position(pos_info['primary_position']),
            'detailed_position': get_position_category(pos_info['primary_position']),
            'position_frequency': pos_info['position_frequency'],
            'matches_with_position_data': pos_info['matches_played'],
            'num_positions_played': pos_info['num_positions'],
            'all_positions': ', '.join(pos_info['all_positions']),
            'is_versatile': pos_info['num_positions'] > 1
        })
    
    position_df = pd.DataFrame(position_data)
    
    # Merge with ranking dataframe
    enhanced_ranking = ranking_df.merge(position_df, on='player_id', how='left')
    
    # Fill missing positions with 'Unknown'
    enhanced_ranking['primary_position'] = enhanced_ranking['primary_position'].fillna('Unknown')
    enhanced_ranking['simplified_position'] = enhanced_ranking['simplified_position'].fillna('Unknown')
    enhanced_ranking['detailed_position'] = enhanced_ranking['detailed_position'].fillna('Unknown')
    
    # Analysis and statistics
    print("=== POSITION ANALYSIS FOR WPR Z-SCORE RANKINGS ===\n")
    
    print("1. Overall Position Distribution (All Players):")
    print(enhanced_ranking['simplified_position'].value_counts())
    print()
    
    print("2. Detailed Position Distribution (All Players):")
    print(enhanced_ranking['detailed_position'].value_counts())
    print()
    
    print("3. Top 20 WPR Players by Position:")
    top_20 = enhanced_ranking.head(20)
    print(top_20['detailed_position'].value_counts())
    print()
    
    print("4. Average WPR Z-Score by Position:")
    avg_by_position = enhanced_ranking.groupby('simplified_position')['average_z_score'].agg(['mean', 'count']).round(3)
    print(avg_by_position)
    print()
    
    print("5. Most Versatile Players (Top 10):")
    versatile = enhanced_ranking[enhanced_ranking['is_versatile'] == True].head(10)
    print(versatile[['player', 'team', 'average_z_score', 'primary_position', 'all_positions']])
    print()
    
    print("6. Top 10 WPR Players with Positions:")
    print(enhanced_ranking[['player', 'team', 'average_z_score', 'detailed_position', 'primary_position']].head(10))
    
    # Save enhanced ranking
    enhanced_ranking.to_csv('../Results/player_rankings_with_positions.csv', index=False)
    print(f"\nEnhanced ranking saved to 'player_rankings_with_positions.csv'")
    
    # Check for players without position data
    no_position = enhanced_ranking[enhanced_ranking['simplified_position'] == 'Unknown']
    if len(no_position) > 0:
        print(f"\nWarning: {len(no_position)} players have no position data")
        print("Sample players without positions:")
        print(no_position[['player', 'team', 'average_z_score']].head())

Found 64 JSON files in /Users/shawnhan/Desktop/Pioneer/Pioneer_Final_Project/Download_Statsbomb/wwc2023/lineups
Extracted position data for 619 unique players
=== POSITION ANALYSIS FOR WPR Z-SCORE RANKINGS ===

1. Overall Position Distribution (All Players):
simplified_position
Midfielder    218
Defender      202
Forward       154
Goalkeeper     41
Name: count, dtype: int64

2. Detailed Position Distribution (All Players):
detailed_position
Fullback/Wing-Back      210
Defensive Midfielder     96
Central Midfielder       84
Forward                  80
Winger                   74
Goalkeeper               41
Attacking Midfielder     30
Name: count, dtype: int64

3. Top 20 WPR Players by Position:
detailed_position
Forward                 8
Central Midfielder      4
Attacking Midfielder    3
Winger                  3
Defensive Midfielder    2
Name: count, dtype: int64

4. Average WPR Z-Score by Position:
                      mean  count
simplified_position              
Defender          